# Agent-handoff surprisal

One metric, one graph: mean fixed-GPT-2 **bits per Unicode character of handoff prose**, plotted against model family release dates with 95% paired task-bootstrap intervals.

Higher means less predictable to this reference model, **not necessarily less intelligible to humans**. This is controlled, initial-investigation communication—not raw CoT, a long-running agent trace, or a loss-of-control measure.

**Run All never calls a paid model API.** The first nonempty scoring run may download pinned GPT-2 weights from Hugging Face; inference is local. Keep raw messages private. No paid collection has been performed merely by opening this notebook.


In [ ]:
from datetime import date, timedelta
from importlib.metadata import version
import hashlib
import json
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "scripts/prepare_tasks.py").exists())
sys.path.insert(0, str(ROOT / "notebooks"))
import surprisal

# Change only these settings to analyze another collected run.
RUN_DIR = ROOT / "data/raw/pilot"
OUT_DIR = ROOT / "out/analysis-pilot"
DEVICE = "auto"  # Apple GPU / CUDA when available; "cpu" is also supported.
RESAMPLES = 10_000
SEED = 0


## 1. Read the raw run and show coverage

The frozen run manifest defines **all** expected task/model pairs, including unattempted cells. Only one complete `spawn_agent(message=...)` call qualifies. API errors, interrupted attempts, truncated responses, malformed calls, and code-only messages remain visible and receive no score—not zero. A changed saved request fails loudly.

The model catalog comes from the run itself, not today's editable configuration.


In [ ]:
def load_run(directory):
    directory = Path(directory)
    if not (directory / "run.json").exists():
        if list(directory.glob("request-*.json")):
            raise ValueError("Raw requests exist without a run manifest")
        return None, []
    manifest = json.loads((directory / "run.json").read_text(encoding="utf-8"))
    expected = manifest["requests"]
    ids = {r["id"] for r in expected}
    if len(ids) != len(expected):
        raise ValueError("Duplicate requests in run manifest")
    extras = {p.stem.removeprefix("request-") for p in directory.glob("request-*.json")} - ids
    if extras:
        raise ValueError("Unexpected requests in this run directory")
    rows = []
    for spec in expected:
        path = directory / f"request-{spec['id']}.json"
        record = json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}
        if path.exists() and any(record.get(k) != v for k, v in spec.items()):
            raise ValueError(f"Saved request disagrees with manifest: {spec['id']}")
        rows.append(extract_handoff(spec, record))
    return manifest, rows


def extract_handoff(spec, record):
    response = record.get("response") or {}
    usage = response.get("usage") or {}
    row = {
        "id": spec["id"], "task_id": spec["task_id"], "model_id": spec["model_id"],
        "context_hash": spec["context_hash"], "repo": spec["source"]["repo"],
        "protocol": spec["protocol"], "split": spec["split"],
        "parameters": json.dumps({k: spec["request"][k] for k in ("reasoning", "max_output_tokens")}, sort_keys=True),
        "served_model": response.get("model"), "collected_at": record.get("started_at"),
        "input_tokens": usage.get("input_tokens"), "output_tokens": usage.get("output_tokens"),
        "reasoning_tokens": (usage.get("output_tokens_details") or {}).get("reasoning_tokens"),
        "status": record.get("state", "not_attempted"), "text": None, "prose": None,
        "bits_per_char": None, "reference_device": None,
    }
    if row["status"] != "finished":
        return row
    if response.get("status") != "completed":
        row["status"] = "incomplete_response"
        return row
    if not isinstance(row["served_model"], str) or not row["served_model"]:
        row["status"] = "missing_model_identity"
        return row
    calls = [o for o in response.get("output", []) if o.get("type") == "function_call"]
    try:
        if len(calls) != 1 or calls[0].get("name") != "spawn_agent":
            raise ValueError("Expected exactly one spawn_agent call")
        args = json.loads(calls[0]["arguments"])
        if not isinstance(args, dict) or set(args) != {"message"}:
            raise ValueError("Expected only a message field")
        if not isinstance(args["message"], str) or not args["message"].strip():
            raise ValueError("Empty/non-string handoff")
        row.update(status="ok", text=args["message"])
    except (ValueError, KeyError, TypeError):
        row["status"] = "invalid_handoff"
    return row


In [ ]:
manifest, rows = load_run(RUN_DIR)
models = manifest["models"] if manifest else json.loads((ROOT / "data/models.json").read_text())
if not manifest:
    print("No collected run found. Prepare tasks, preview collection, then explicitly run the paid collector.")
    print("No reference model will be loaded and no model results will be invented.")

columns = ["id", "task_id", "model_id", "context_hash", "repo", "protocol", "split",
           "parameters", "served_model", "collected_at", "input_tokens", "output_tokens",
           "reasoning_tokens", "status", "text", "prose", "bits_per_char", "reference_device"]
samples = pd.DataFrame(rows, columns=columns)
display(pd.crosstab(samples["model_id"], samples["status"]).reindex(
    [m["id"] for m in models], fill_value=0))
if not samples.empty:
    display(samples.groupby("model_id")[["input_tokens", "output_tokens", "reasoning_tokens"]].sum(min_count=1))


## 2. Score original prose locally

For each message: **BPC = total negative log₂ probability of its reference tokens / Unicode characters in its prose.**

The small helper pins GPT-2 and its tokenizer, uses BOS for the first token, and scores each token once with overlapping 1,024-token windows (stride 512). It replaces fenced/inline code, URLs, and common path forms with a single space and trims outer whitespace. It does **not** repair spacing, paraphrase, or judge meaning. Bare identifiers and unfenced code can remain; inspect examples below.

All retained spaces and punctuation count in the denominator. Empty/all-code messages are missing, never zero. Scores are recomputed from saved raw text; there is no opaque measurement cache.


In [ ]:
for index, row in samples.iterrows():
    if row["status"] != "ok":
        continue
    prose = surprisal.prose_only(row["text"])
    samples.at[index, "prose"] = prose
    if not prose:
        samples.at[index, "status"] = "no_prose"
        continue
    # Errors in local scoring stop the notebook rather than silently dropping examples.
    result = surprisal.bits_per_char(prose, device=DEVICE)
    samples.at[index, "bits_per_char"] = result["bits_per_char"]
    samples.at[index, "reference_device"] = result.get("device")

samples["bits_per_char"] = pd.to_numeric(samples["bits_per_char"])
display(pd.crosstab(samples["model_id"], samples["status"]).reindex(
    [m["id"] for m in models], fill_value=0))


## 3. Match tasks and bootstrap whole tasks

Only tasks scored for **every requested model** enter the graph. Each task has equal weight. Resampling the same task indices across models preserves pairing; tokens and snippets are not independent samples.

Intervals are the 2.5th/97.5th percentiles of 10,000 resampled means. One task gets no interval; fewer than ten is flagged as a pilot. Missingness can bias the retained sample. These nominal intervals describe variation across this selected task set, not validation of the metric or all agent workloads.

The study is approximately repository-balanced, not prevalence-weighted software usage. Related tasks within a repository may remain dependent. A model's settings and returned model identity must be consistent within the run.


In [ ]:
def summarize(samples, models, resamples=10_000, seed=0):
    if resamples < 1:
        raise ValueError("resamples must be positive")
    ids = [m["id"] for m in models]
    if not ids or len(ids) != len(set(ids)):
        raise ValueError("Need a nonempty, unique model catalog")
    for model in models:
        date.fromisoformat(model["release_date"])
        if not model["release_source"].startswith("https://"):
            raise ValueError("Release dates need source URLs")
    if samples.duplicated(["task_id", "model_id"]).any():
        raise ValueError("Expected one observation per task/model")
    if not set(samples["model_id"]).issubset(ids):
        raise ValueError("Unknown model in samples")
    if samples["protocol"].nunique() > 1 or samples["split"].nunique() > 1:
        raise ValueError("Do not combine protocols or pilot/main splits")
    if (samples.groupby("task_id")["context_hash"].nunique() > 1).any():
        raise ValueError("A task ID refers to different contexts")
    for model_id, group in samples.groupby("model_id"):
        if group["parameters"].nunique() > 1 or group["served_model"].dropna().nunique() > 1:
            raise ValueError(f"Mixed settings or served snapshots: {model_id}")
    valid = samples.loc[samples["status"].eq("ok")].copy()
    if not valid.empty and (not np.isfinite(valid["bits_per_char"].to_numpy(dtype=float)).all()
                            or (valid["bits_per_char"] < 0).any()):
        raise ValueError("Invalid reference scores")
    matrix = valid.pivot(index="task_id", columns="model_id", values="bits_per_char").reindex(columns=ids)
    matrix = matrix.dropna().sort_index()
    points = []
    if len(matrix):
        values = matrix.to_numpy(dtype=float)
        means = values.mean(axis=0)
        lower = upper = [None] * len(models)
        if len(matrix) > 1:
            indices = np.random.default_rng(seed).integers(0, len(matrix), size=(resamples, len(matrix)))
            draws = values[indices].mean(axis=1)
            lower, upper = np.quantile(draws, [.025, .975], axis=0)
        for j, model in enumerate(models):
            points.append({**model, "mean_bpc": float(means[j]),
                           "ci_low": float(lower[j]) if lower[j] is not None else None,
                           "ci_high": float(upper[j]) if upper[j] is not None else None,
                           "n_tasks": len(matrix)})
    return {"points": points, "task_ids": matrix.index.tolist(),
            "task_scores": json.loads(matrix.to_json(orient="index")),
            "n_planned_tasks": int(samples["task_id"].nunique()),
            "n_matched_tasks": len(matrix), "resamples": resamples, "seed": seed,
            "confidence_level": .95, "ci_method": "paired task percentile bootstrap"}


In [ ]:
summary = summarize(samples, models, RESAMPLES, SEED)
print(f"Matched tasks: {summary['n_matched_tasks']} / {summary['n_planned_tasks']} planned")
if 0 < summary["n_matched_tasks"] < 10:
    print("Pilot-sized sample: interpret intervals cautiously.")
if summary["n_matched_tasks"] == 1:
    print("One task cannot support a bootstrap interval.")
display(pd.DataFrame(summary["points"]).reindex(
    columns=["label", "mean_bpc", "ci_low", "ci_high", "n_tasks"]))


## 4. The single result graph

Dates index **model-family public releases**, not when requests were collected and not necessarily when a returned alias snapshot launched. This is a comparison collected now, not historical monitoring. No fitted trend line is implied by three models.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
color = "#326a88"
for i, point in enumerate(summary["points"]):
    when = date.fromisoformat(point["release_date"])
    lo, hi = point["ci_low"], point["ci_high"]
    if lo is not None:
        # A percentile interval need not be symmetric around the observed mean.
        ax.errorbar(when, (lo + hi) / 2, yerr=(hi - lo) / 2,
                    fmt="none", capsize=5, color=color)
    ax.plot(when, point["mean_bpc"], "o", color=color)
    ax.annotate(point["label"], (when, point["mean_bpc"]), xytext=(8, 12 if i % 2 == 0 else -22),
                textcoords="offset points", fontsize=10)
dates = [date.fromisoformat(m["release_date"]) for m in models]
padding = timedelta(days=max(21, (max(dates) - min(dates)).days * .2))
ax.set_xlim(min(dates) - padding, max(dates) + padding)
if not summary["points"]:
    ax.text(.5, .5, "No comparable model scores yet", transform=ax.transAxes, ha="center")
split = manifest["split"] if manifest else "not collected"
ax.set(title=f"Agent-handoff surprisal · {split}",
       xlabel="Model-family public-release date", ylabel="Mean surprisal (bits / character of prose)")
locator = mdates.AutoDateLocator(minticks=3, maxticks=6)
ax.xaxis.set_major_locator(locator)
ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator))
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=.2)
note = f"95% task-bootstrap intervals · {summary['n_matched_tasks']} matched tasks · higher = less predictable text"
if 0 < summary["n_matched_tasks"] < 10:
    note += "\nPilot sample; intervals are provisional."
fig.text(.5, .025, note, ha="center", fontsize=9)
fig.tight_layout(rect=(0, .09, 1, 1))
plt.show()


## 5. Inspect examples and save derived results

Inspect a few low-, middle-, and high-scoring messages per model. Does the metric track the behavior you care about, or mostly jargon/formatting? This is a qualitative sanity check, not an additional metric. Do not retune the pipeline to obtain a preferred model ordering.

Exports go only to `out/`. Exact inputs/responses in `data/raw/` remain untouched. The source messages shown and exported here can contain private information.


In [ ]:
for model_id, group in samples.loc[samples["status"].eq("ok")].groupby("model_id"):
    ordered = group.sort_values("bits_per_char")
    positions = sorted({0, len(ordered) // 2, len(ordered) - 1})
    print(model_id)
    with pd.option_context("display.max_colwidth", 180):
        display(ordered.iloc[positions][["task_id", "bits_per_char", "text", "prose"]])

OUT_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(OUT_DIR / "surprisal.png", dpi=180, bbox_inches="tight")
samples.to_json(OUT_DIR / "samples.jsonl", orient="records", lines=True, force_ascii=False)
summary["models"] = models
summary["reference"] = {"model": surprisal.MODEL, "revision": surprisal.REVISION,
                        "scorer_version": surprisal.VERSION, "stride": surprisal.STRIDE,
                        "device_requested": DEVICE,
                        "devices_used": sorted(samples["reference_device"].dropna().unique().tolist()),
                        "prose_exclusion_regex": surprisal.NONPROSE.pattern,
                        "code_sha256": hashlib.sha256(Path(surprisal.__file__).read_bytes()).hexdigest()}
summary["software"] = {name: version(name) for name in ("torch", "transformers", "numpy", "pandas", "matplotlib")}
summary["run_hash"] = hashlib.sha256(json.dumps(manifest, sort_keys=True, ensure_ascii=False).encode()).hexdigest() if manifest else None
summary["coverage"] = [{"model_id": m, "status": s, "count": int(n)}
                       for (m, s), n in samples.groupby(["model_id", "status"]).size().items()]
(OUT_DIR / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2, allow_nan=False) + "\n", encoding="utf-8")
print(f"Saved PNG, per-message data, and exact plot summary to {OUT_DIR}")
